# Chapitre 7 — Apprentissage auto-supervisé et JEPA
## Atelier corrigé · M2 IA
PyTorch CPU, scikit-learn (jeu digits inclus), Matplotlib.


In [ ]:
import math, time, copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
torch.set_num_threads(4)
print('PyTorch :', torch.__version__)


## Atelier 1 · Données, vues et test décalé


In [ ]:
# CELLULE 2 — Données : chiffres 8 × 8, sans étiquettes pour l'apprentissage auto-supervisé.
digits = load_digits()
X_all = torch.tensor(digits.images / 16., dtype=torch.float32)          # (1797, 8, 8) dans [0, 1]
y_all = torch.tensor(digits.target)
perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(0))
X_ssl, y_ssl = X_all[perm[:1200]], y_all[perm[:1200]]   # étiquettes ignorées pendant le préentraînement
X_test, y_test = X_all[perm[1200:]], y_all[perm[1200:]]
def draw_few(seed, per_class=5):
    g = torch.Generator().manual_seed(seed)
    return torch.cat([torch.nonzero(y_ssl == c)[:, 0][torch.randperm(int((y_ssl == c).sum()), generator=g)[:per_class]] for c in range(10)])
FEW = [draw_few(s) for s in range(5)]                    # cinq tirages de 5 étiquettes par classe
# Test « décalé » : chaque chiffre translaté d’un pixel en diagonale — l’invariance que les vues ont enseignée.
g_shift = torch.Generator().manual_seed(7)
shifts = 2 * torch.randint(0, 2, (len(X_test), 2), generator=g_shift) - 1
X_test_shift = torch.stack([torch.roll(x, (int(s[0]), int(s[1])), (0, 1)) for x, s in zip(X_test, shifts)])
print('Auto-supervisé :', len(X_ssl), '; sondes à 50 étiquettes (5 tirages) ; test :', len(X_test), '(+ version décalée)')

def augment(x, gen):
    """Deux vues d'un même chiffre : translation, effacement d'un bloc, intensité, bruit."""
    B = x.shape[0]
    dx, dy = torch.randint(-1, 2, (2, B), generator=gen)
    x = torch.stack([torch.roll(x[i], (int(dy[i]), int(dx[i])), (0, 1)) for i in range(B)])
    i0, j0 = torch.randint(0, 6, (2, B), generator=gen)
    erase = torch.rand(B, generator=gen) < .5
    for b in torch.nonzero(erase)[:, 0]:
        x[b, i0[b]:i0[b] + 3, j0[b]:j0[b] + 3] = 0.
    x = x * (.7 + .6 * torch.rand(B, 1, 1, generator=gen))
    return (x + .05 * torch.randn(x.shape, generator=gen)).clamp(0, 1)


### Encodeur et trois mesures : sonde, k-NN, rang effectif


In [ ]:
# CELLULE 3 — Encodeur, projecteur, et trois mesures des représentations.
def encoder():
    return nn.Sequential(nn.Flatten(), nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 64))
def projector(d_in=64, d=128):
    return nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))

@torch.no_grad()
def embed(enc, X):
    enc.eval(); Z = enc(X); enc.train(); return Z

def linear_probe(Z_tr, y_tr, Z_te, y_te, steps=500):
    mu, sd = Z_tr.mean(0), Z_tr.std(0) + 1e-6
    head = nn.Linear(Z_tr.shape[1], 10)
    opt = torch.optim.Adam(head.parameters(), lr=1e-2, weight_decay=1e-3)
    for _ in range(steps):
        loss = F.cross_entropy(head((Z_tr - mu) / sd), y_tr)
        opt.zero_grad(); loss.backward(); opt.step()
    return (head((Z_te - mu) / sd).argmax(1) == y_te).float().mean().item()

def knn(Z_tr, y_tr, Z_te, y_te, k=5):
    A, B = F.normalize(Z_tr, dim=1), F.normalize(Z_te, dim=1)
    idx = (B @ A.T).topk(k, dim=1).indices
    return (y_tr[idx].mode(1).values == y_te).float().mean().item()

def effective_rank(Z):
    s = torch.linalg.svdvals(Z - Z.mean(0))
    p = s / s.sum()
    return float(torch.exp(-(p * torch.log(p + 1e-12)).sum()))

def evaluate(enc):
    Ztr, Zte, Zsh = embed(enc, X_ssl), embed(enc, X_test), embed(enc, X_test_shift)
    p50 = [linear_probe(Ztr[f], y_ssl[f], Zte, y_test) for f in FEW]
    return dict(probe50=float(np.mean(p50)), probe50_sd=float(np.std(p50)),
                probe_all=linear_probe(Ztr, y_ssl, Zte, y_test),
                shift_all=linear_probe(Ztr, y_ssl, Zsh, y_test),
                knn=knn(Ztr, y_ssl, Zte, y_test), rank=effective_rank(Ztr),
                spectrum=torch.linalg.svdvals(Ztr - Ztr.mean(0)).tolist())


## Atelier 2 · Invariance, InfoNCE, VICReg
Prédisez lequel de ces objectifs s’effondre (TD, exercice 3).


In [ ]:
# CELLULE 4 — Quatre objectifs sur deux vues : invariance seule, InfoNCE, VICReg, VICReg sans variance.
def info_nce(z1, z2, tau=.2):
    z = F.normalize(torch.cat([z1, z2]), dim=1)
    sim = z @ z.T / tau
    sim.fill_diagonal_(-float('inf'))                 # un exemple n'est pas son propre négatif
    n = z1.shape[0]
    target = torch.cat([torch.arange(n, 2 * n), torch.arange(n)])
    return F.cross_entropy(sim, target)

def vicreg(z1, z2, lam=25., mu=25., nu=1., use_variance=True):
    inv = F.mse_loss(z1, z2)
    var = cov = 0.
    for z in (z1, z2):
        z = z - z.mean(0)
        if use_variance:
            var = var + F.relu(1 - torch.sqrt(z.var(0) + 1e-4)).mean()
        c = (z.T @ z) / (len(z) - 1)
        cov = cov + (c - torch.diag(torch.diag(c))).pow(2).sum() / z.shape[1]
    return lam * inv + mu * var + nu * cov

def train_joint(loss_name, steps=1500, seed=0):
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    enc, proj = encoder(), projector()
    opt = torch.optim.Adam(list(enc.parameters()) + list(proj.parameters()), lr=1e-3)
    history = []
    for step in range(steps):
        x = X_ssl[torch.randint(0, len(X_ssl), (256,), generator=gen)]
        z1, z2 = proj(enc(augment(x, gen))), proj(enc(augment(x, gen)))
        if loss_name == 'invariance':
            loss = F.mse_loss(z1, z2)
        elif loss_name == 'infonce':
            loss = info_nce(z1, z2)
        else:
            loss = vicreg(z1, z2, use_variance=(loss_name == 'vicreg'))
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 100 == 0:
            history.append((step, effective_rank(embed(enc, X_ssl[:512]))))
    return enc, history


### Sans négatifs : BYOL et une mini I-JEPA


In [ ]:
# CELLULE 5 — Sans négatifs : prédicteur + cible à moyenne mobile (BYOL), puis une mini I-JEPA.
def ema_update(target, online, m):
    with torch.no_grad():
        for pt, po in zip(target.parameters(), online.parameters()):
            pt.mul_(m).add_(po, alpha=1 - m)

def train_byol(steps=1500, seed=0, m=.99):
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    enc, proj = encoder(), projector()
    pred = nn.Sequential(nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128, 128))
    t_enc, t_proj = copy.deepcopy(enc), copy.deepcopy(proj)
    for p in list(t_enc.parameters()) + list(t_proj.parameters()):
        p.requires_grad_(False)
    opt = torch.optim.Adam(list(enc.parameters()) + list(proj.parameters()) + list(pred.parameters()), lr=1e-3)
    history = []
    for step in range(steps):
        x = X_ssl[torch.randint(0, len(X_ssl), (256,), generator=gen)]
        v1, v2 = augment(x, gen), augment(x, gen)
        p1, p2 = pred(proj(enc(v1))), pred(proj(enc(v2)))
        with torch.no_grad():
            t1, t2 = t_proj(t_enc(v1)), t_proj(t_enc(v2))      # cibles : pas de gradient
        loss = (2 - 2 * F.cosine_similarity(p1, t2).mean()) + (2 - 2 * F.cosine_similarity(p2, t1).mean())
        opt.zero_grad(); loss.backward(); opt.step()
        ema_update(t_enc, enc, m); ema_update(t_proj, proj, m)
        if step % 100 == 0:
            history.append((step, effective_rank(embed(enc, X_ssl[:512]))))
    return enc, history

QUADS = [(0, 0), (0, 4), (4, 0), (4, 4)]
def block_mask(q):
    m = torch.zeros(8, 8); i, j = QUADS[q]; m[i:i + 4, j:j + 4] = 1.; return m

def train_ijepa(steps=1500, seed=0, m=.99):
    """Contexte : l'image privée d'un quart. Cible : la représentation (EMA) de ce quart seul."""
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    ctx_enc = encoder()
    tgt_enc = copy.deepcopy(ctx_enc)
    for p in tgt_enc.parameters():
        p.requires_grad_(False)
    pred = nn.Sequential(nn.Linear(64 + 4, 256), nn.ReLU(), nn.Linear(256, 64))   # conditionné par la position
    opt = torch.optim.Adam(list(ctx_enc.parameters()) + list(pred.parameters()), lr=1e-3)
    masks = torch.stack([block_mask(q) for q in range(4)])
    history = []
    for step in range(steps):
        x = X_ssl[torch.randint(0, len(X_ssl), (256,), generator=gen)]
        q = torch.randint(0, 4, (len(x),), generator=gen)
        mk = masks[q]
        s_x = ctx_enc(x * (1 - mk))                                   # contexte
        with torch.no_grad():
            s_y = tgt_enc(x * mk)                                     # cible dans l'espace latent
        s_hat = pred(torch.cat([s_x, F.one_hot(q, 4).float()], 1))
        loss = F.mse_loss(F.layer_norm(s_hat, (64,)), F.layer_norm(s_y, (64,)))
        opt.zero_grad(); loss.backward(); opt.step()
        ema_update(tgt_enc, ctx_enc, m)
        if step % 100 == 0:
            history.append((step, effective_rank(embed(ctx_enc, X_ssl[:512]))))
    return ctx_enc, history


## Atelier 3 · Évaluation commune (≈ 4 min)
Comparez aussi au test décalé d’un pixel.


In [ ]:
# CELLULE 6 — Tout comparer avec les mêmes sondes, plus deux références.
results, histories = {}, {}
torch.manual_seed(0)
results['aléatoire'] = evaluate(encoder())
results['pixels'] = evaluate(nn.Flatten())             # l'« encodeur » identité
for name in ['invariance', 'vicreg_sans_variance', 'infonce', 'vicreg']:
    t0 = time.time()
    enc, histories[name] = train_joint(name)
    results[name] = evaluate(enc)
    print(f'{name:22s} ({time.time()-t0:.0f} s)')
for name, fn in [('byol', train_byol), ('ijepa', train_ijepa)]:
    t0 = time.time()
    enc, histories[name] = fn()
    results[name] = evaluate(enc)
    print(f'{name:22s} ({time.time()-t0:.0f} s)')
print(f'\n{"méthode":22s} sonde 50 étiq.   sonde 1200  décalé   k-NN   rang eff.')
for k, r in results.items():
    print(f'{k:22s} {r["probe50"]:.3f} ± {r["probe50_sd"]:.3f}   {r["probe_all"]:.3f}       {r["shift_all"]:.3f}    {r["knn"]:.3f}  {r["rank"]:.1f}')


## Atelier 4 · Une JEPA pour des séquences
Le langage à quatre thèmes du chapitre 6 : prédire la représentation du futur, pas ses jetons.


In [ ]:
# CELLULE 7 — JEPA pour des séquences : prédire la représentation de la suite, pas ses jetons.
# On reprend le langage à quatre thèmes du chapitre 6 (même générateur).
V, T, K = 24, 32, 4
g = torch.Generator().manual_seed(0)
TRANS = (2.0 * torch.randn(V, V, generator=g) + 1.5 * torch.randn(K, V, V, generator=g)).softmax(-1)
def sample(n, seed):
    gen = torch.Generator().manual_seed(seed)
    z = torch.randint(0, K, (n,), generator=gen)
    x = torch.empty(n, T, dtype=torch.long); x[:, 0] = torch.randint(0, V, (n,), generator=gen)
    for t in range(1, T):
        x[:, t] = torch.multinomial(TRANS[z, x[:, t - 1]], 1, generator=gen).squeeze(1)
    return x, z
S_pre, _ = sample(20000, 1); S_tr, zs_tr = sample(64, 3); S_te, zs_te = sample(4000, 4)

class SeqEncoder(nn.Module):
    """Sac de transitions (x_t, x_{t+1}) plongées, puis MLP : assez pour capter un thème."""
    def __init__(self, d=64):
        super().__init__()
        self.bag = nn.EmbeddingBag(V * V, 64, mode='mean')
        self.mlp = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, d))
    def forward(self, x):
        return self.mlp(self.bag(x[:, :-1] * V + x[:, 1:]))

def seq_jepa(mode='ema', steps=1500, seed=0, m=.99):
    """mode : 'ema' (cible EMA sans gradient), 'naive' (cible = encodeur en ligne, avec gradient)."""
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    ctx = SeqEncoder(); tgt = copy.deepcopy(ctx)
    for p in tgt.parameters():
        p.requires_grad_(False)
    pred = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64))
    opt = torch.optim.Adam(list(ctx.parameters()) + list(pred.parameters()), lr=1e-3)
    for step in range(steps):
        x = S_pre[torch.randint(0, len(S_pre), (256,), generator=gen)]
        past, future = x[:, :16], x[:, 16:]                   # contexte : 16 jetons ; cible : les 16 suivants
        s_hat = pred(ctx(past))
        s_y = ctx(future) if mode == 'naive' else tgt(future).detach()
        loss = F.mse_loss(F.normalize(s_hat, dim=1), F.normalize(s_y, dim=1))
        opt.zero_grad(); loss.backward(); opt.step()
        if mode == 'ema':
            ema_update(tgt, ctx, m)
    return ctx

seq_results = {}
for name, enc in [('aléatoire', SeqEncoder()), ('JEPA naïf (sans EMA)', seq_jepa('naive')), ('JEPA (cible EMA)', seq_jepa('ema'))]:
    with torch.no_grad():
        Ztr, Zte = enc(S_tr), enc(S_te); Zall = enc(S_pre[:2000])
    seq_results[name] = (linear_probe(Ztr, zs_tr, Zte, zs_te), effective_rank(Zall))
    print(f'{name:22s} thème, sonde 64 étiquettes : {seq_results[name][0]:.3f} ; rang effectif {seq_results[name][1]:.1f}')
print('Référence chapitre 6 : modèle de langage autorégressif + sonde = 0,978')
